# 82: Momentum-Based Exit Filters

**Problem:** Valuation exits (STH-MVRV, Z-scores) fire too early. "Never Exit" keeps winning.

**Root Cause:** We exit when BTC is "expensive" but momentum is still strong. 2023-2026 shows sustained uptrend with high valuations.

## The Solution: Momentum Filters

**Don't exit just because it's expensive. Exit when:**
1. Valuation is high (STH-MVRV > 1.5 OR MVRV > 2.0)
2. **AND** momentum is weakening (trend reversal)

This keeps you invested during strong uptrends, even when "expensive".

## Momentum Indicators to Test:

| Indicator | What It Measures | Exit Signal |
|-----------|------------------|-------------|
| **Price vs MA** | Trend direction | Price < 50-day MA (downtrend) |
| **MA Cross** | Trend change | 50-day MA < 200-day MA (death cross) |
| **RSI** | Overbought/oversold | RSI > 80 then < 70 (momentum fade) |
| **MACD** | Momentum shift | MACD crosses below signal line |
| **ATR Breakout** | Volatility expansion | Price breaks key support levels |
| **ROC** | Rate of change | 30-day ROC turns negative |

## Exit Strategies to Test:

1. **Never Exit** (baseline)
2. **Valuation Only** - STH-MVRV > 1.5 (fails, we know this)
3. **Momentum Only** - Price < 50-day MA
4. **Combined:** STH-MVRV > 1.5 AND Price < 50-day MA
5. **Combined:** STH-MVRV > 1.5 AND MACD bearish cross
6. **Combined:** MVRV > 2.0 AND Death cross (50MA < 200MA)
7. **Aggressive momentum:** Price < 20-day MA (faster exits)
8. **Conservative momentum:** Price < 100-day MA (patient exits)

**Hypothesis:** Combining valuation + momentum should beat "Never Exit" by avoiding exits during strong trends.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
df.head()

## 2. Calculate Momentum Indicators

In [ ]:
print("Calculating momentum indicators...\n")

price = df['price']

# 1. Moving Averages
df['ma_20'] = price.rolling(20).mean()
df['ma_50'] = price.rolling(50).mean()
df['ma_100'] = price.rolling(100).mean()
df['ma_200'] = price.rolling(200).mean()

# 2. Price position relative to MAs
df['above_ma_20'] = price > df['ma_20']
df['above_ma_50'] = price > df['ma_50']
df['above_ma_100'] = price > df['ma_100']
df['above_ma_200'] = price > df['ma_200']

# 3. MA crossovers
df['golden_cross'] = df['ma_50'] > df['ma_200']  # Bullish
df['death_cross'] = df['ma_50'] < df['ma_200']   # Bearish

# 4. RSI (14-period)
delta = price.diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

# 5. MACD (12, 26, 9)
ema_12 = price.ewm(span=12, adjust=False).mean()
ema_26 = price.ewm(span=26, adjust=False).mean()
df['macd'] = ema_12 - ema_26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']
df['macd_bullish'] = df['macd'] > df['macd_signal']
df['macd_bearish'] = df['macd'] < df['macd_signal']

# 6. Rate of Change (ROC)
df['roc_30'] = price.pct_change(30)
df['roc_60'] = price.pct_change(60)
df['roc_90'] = price.pct_change(90)

# 7. ATR-based trend strength
returns = price.pct_change()
df['volatility'] = returns.rolling(30).std() * np.sqrt(365)

print("✓ Momentum indicators calculated")
print(f"\nCurrent values (latest day):")
print(f"  Price: ${price.iloc[-1]:,.0f}")
print(f"  MA-50: ${df['ma_50'].iloc[-1]:,.0f} ({'above' if df['above_ma_50'].iloc[-1] else 'below'})")
print(f"  MA-200: ${df['ma_200'].iloc[-1]:,.0f} ({'above' if df['above_ma_200'].iloc[-1] else 'below'})")
print(f"  RSI: {df['rsi'].iloc[-1]:.1f}")
print(f"  MACD: {'Bullish' if df['macd_bullish'].iloc[-1] else 'Bearish'}")
print(f"  30d ROC: {df['roc_30'].iloc[-1]:.1%}")

## 3. Visualize Price vs Momentum

In [ ]:
# Plot price with momentum indicators
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# 1. Price with MAs
ax1 = axes[0]
ax1.plot(df.index, df['price'], label='Price', linewidth=2, color='black')
ax1.plot(df.index, df['ma_50'], label='50-day MA', linewidth=1.5, color='blue', alpha=0.7)
ax1.plot(df.index, df['ma_200'], label='200-day MA', linewidth=1.5, color='red', alpha=0.7)
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['death_cross']), alpha=0.2, color='red', label='Death Cross')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title('Price vs Moving Averages', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. RSI
ax2 = axes[1]
ax2.plot(df.index, df['rsi'], label='RSI', linewidth=1.5, color='purple')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.fill_between(df.index, 0, 100, where=(df['rsi'] > 70), alpha=0.2, color='red')
ax2.fill_between(df.index, 0, 100, where=(df['rsi'] < 30), alpha=0.2, color='green')
ax2.set_ylabel('RSI', fontsize=12)
ax2.set_ylim(0, 100)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. MACD
ax3 = axes[2]
ax3.plot(df.index, df['macd'], label='MACD', linewidth=1.5, color='blue')
ax3.plot(df.index, df['macd_signal'], label='Signal', linewidth=1.5, color='red')
ax3.bar(df.index, df['macd_hist'], label='Histogram', alpha=0.3, color='gray')
ax3.axhline(0, color='black', linestyle='-', alpha=0.3)
ax3.set_ylabel('MACD', fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# 4. ROC
ax4 = axes[3]
ax4.plot(df.index, df['roc_30'], label='30-day ROC', linewidth=1.5, color='green')
ax4.axhline(0, color='black', linestyle='-', alpha=0.3)
ax4.fill_between(df.index, -1, 1, where=(df['roc_30'] < 0), alpha=0.2, color='red')
ax4.set_ylabel('30d ROC', fontsize=12)
ax4.set_xlabel('Date', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Momentum indicators over time")

## 4. Generate Entry Signals

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
print("Generating entry signals...\n")

c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0

c4 = c4.fillna(False)
c5 = c5.fillna(False)

entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = (entry_count >= 4).fillna(False).astype(bool)

print(f"✓ Entry signals: {entries.sum()}")

## 5. Generate Exit Signals (Momentum-Based)

In [ ]:
print("Generating momentum-based exit signals...\n")

# Valuation signals
sth_overheated = (df['mvrv_sth'] > 1.5).fillna(False)
mvrv_overheated = (df['mvrv'] > 2.0).fillna(False)
lth_distribution = (df['sopr_lth'] > 1.5).fillna(False)

# Momentum signals (bearish)
below_ma_20 = (~df['above_ma_20']).fillna(False)
below_ma_50 = (~df['above_ma_50']).fillna(False)
below_ma_100 = (~df['above_ma_100']).fillna(False)
below_ma_200 = (~df['above_ma_200']).fillna(False)
death_cross = (df['death_cross']).fillna(False)
macd_bearish = (df['macd_bearish']).fillna(False)
roc_negative = (df['roc_30'] < 0).fillna(False)

exit_strategies = {}

# Baseline
exit_strategies['never'] = (pd.Series(False, index=df.index, dtype=bool), 'Never Exit')

# Pure valuation (we know these fail)
exit_strategies['sth_only'] = (sth_overheated.astype(bool), 'STH-MVRV > 1.5 only')
exit_strategies['mvrv_only'] = (mvrv_overheated.astype(bool), 'MVRV > 2.0 only')

# Pure momentum
exit_strategies['ma50'] = (below_ma_50.astype(bool), 'Price < 50MA')
exit_strategies['ma200'] = (below_ma_200.astype(bool), 'Price < 200MA')
exit_strategies['death_cross'] = (death_cross.astype(bool), 'Death Cross')
exit_strategies['macd'] = (macd_bearish.astype(bool), 'MACD Bearish')

# Combined: Valuation + Momentum (KEY TESTS)
exit_strategies['sth_ma50'] = ((sth_overheated & below_ma_50).astype(bool), 'STH>1.5 AND Price<50MA')
exit_strategies['sth_ma100'] = ((sth_overheated & below_ma_100).astype(bool), 'STH>1.5 AND Price<100MA')
exit_strategies['sth_macd'] = ((sth_overheated & macd_bearish).astype(bool), 'STH>1.5 AND MACD Bearish')
exit_strategies['sth_roc'] = ((sth_overheated & roc_negative).astype(bool), 'STH>1.5 AND ROC<0')

exit_strategies['mvrv_ma50'] = ((mvrv_overheated & below_ma_50).astype(bool), 'MVRV>2.0 AND Price<50MA')
exit_strategies['mvrv_death'] = ((mvrv_overheated & death_cross).astype(bool), 'MVRV>2.0 AND Death Cross')

# Original Check framework
exit_strategies['original'] = (((df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)).fillna(False).astype(bool), 'Original (MVRV>2.0 AND LTH-SOPR>1.5)')

# Aggressive: Exit on any momentum weakness
exit_strategies['ma20'] = (below_ma_20.astype(bool), 'Price < 20MA (aggressive)')

print("Exit signal counts:")
print("="*80)
for key, (exits, name) in exit_strategies.items():
    print(f"{name:<50} {exits.sum():>4} signals")

# Verify dtypes
print("\nDtype verification:")
for key, (exits, name) in list(exit_strategies.items())[:3]:
    print(f"  {key}: {exits.dtype}")

## 6. Backtest All Strategies

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

print("\n" + "="*90)
print("BACKTESTING: MOMENTUM-BASED EXIT FILTERS")
print("="*90)

# Backtest all strategies
results = {}
for key, (exits, name) in exit_strategies.items():
    results[key] = backtest_strategy(df, entries, exits, name)

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Sort by return
sorted_results = sorted(results.items(), key=lambda x: x[1]['total_return'], reverse=True)

# Results table
print(f"\n{'Strategy':<50} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*90)

for key, res in sorted_results:
    print(f"{res['name']:<50} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<50} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*90)

# Best strategies
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

print(f"\n🏆 BEST RETURN: {best['name']} at {best['total_return']:.1f}%")
print(f"📊 BEST SHARPE: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

# Key comparisons
print("\n🔍 KEY COMPARISONS:")
never = results['never']
print(f"  Never Exit: {never['total_return']:.1f}%")

# Compare combined strategies
for key in ['sth_ma50', 'sth_ma100', 'sth_macd', 'mvrv_ma50']:
    if key in results:
        res = results[key]
        diff = res['total_return'] - never['total_return']
        status = "✅" if diff > 0 else "❌"
        print(f"  {status} {res['name']}: {res['total_return']:.1f}% ({diff:+.1f}% vs Never)")

## 7. Equity Curves (Top Strategies)

In [ ]:
# Plot top 6 strategies
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=3, color='black', linestyle='--', alpha=0.7)

# Top 6 strategies
colors = ['green', 'blue', 'orange', 'red', 'purple', 'cyan']
for (key, res), color in zip(sorted_results[:6], colors):
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=color)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Momentum Exit Filters: Top Strategies', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Performance by Time Period

In [ ]:
# Test by period
periods = [
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY TIME PERIOD")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test key strategies
    period_results = {}
    for key in ['never', 'ma50', 'sth_ma50', 'sth_ma100', 'mvrv_ma50', 'original']:
        if key not in exit_strategies:
            continue
        period_exits = exit_strategies[key][0][(exit_strategies[key][0].index >= start) & (exit_strategies[key][0].index <= end)]
        
        try:
            pf = vbt.Portfolio.from_signals(
                close=period_df['price'], entries=period_entries, exits=period_exits,
                fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
            )
            period_results[key] = pf.total_return() * 100
        except:
            period_results[key] = 0
    
    print(f"\n{label}:")
    print(f"  Buy & Hold: {bh:+.1f}%")
    
    for key, ret in sorted(period_results.items(), key=lambda x: x[1], reverse=True):
        name = exit_strategies[key][1]
        print(f"  {name}: {ret:+.1f}% ({ret - bh:+.1f}% vs B&H)")
    
    best_key = max(period_results, key=period_results.get)
    print(f"  🏆 Winner: {exit_strategies[best_key][1]}")

print("\n" + "="*100)

## 9. Final Verdict

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: MOMENTUM EXIT FILTERS")
print("="*90)

never = results['never']
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

# Get key combined strategies
sth_ma50 = results.get('sth_ma50')
sth_ma100 = results.get('sth_ma100')
mvrv_ma50 = results.get('mvrv_ma50')

print(f"\n1. PERFORMANCE COMPARISON:")
print(f"   Buy & Hold:              {bh_return:.1f}%")
print(f"   Never Exit:              {never['total_return']:.1f}% ({never['total_return'] - bh_return:+.1f}%)")
if sth_ma50:
    print(f"   STH>1.5 AND Price<50MA:  {sth_ma50['total_return']:.1f}% ({sth_ma50['total_return'] - bh_return:+.1f}%)")
if sth_ma100:
    print(f"   STH>1.5 AND Price<100MA: {sth_ma100['total_return']:.1f}% ({sth_ma100['total_return'] - bh_return:+.1f}%)")
if mvrv_ma50:
    print(f"   MVRV>2.0 AND Price<50MA: {mvrv_ma50['total_return']:.1f}% ({mvrv_ma50['total_return'] - bh_return:+.1f}%)")
print(f"\n   🏆 Best Return: {best['name']} at {best['total_return']:.1f}%")
print(f"   📊 Best Sharpe: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

print(f"\n2. KEY INSIGHTS:")

if sth_ma50 and sth_ma100:
    improvement_50 = sth_ma50['total_return'] - never['total_return']
    improvement_100 = sth_ma100['total_return'] - never['total_return']
    
    print(f"   STH+MA50 vs Never Exit:  {improvement_50:+.1f}%")
    print(f"   STH+MA100 vs Never Exit: {improvement_100:+.1f}%")
    
    if improvement_50 > 50 or improvement_100 > 50:
        print(f"   ✅ Momentum filters add MASSIVE value!")
        print(f"   💡 Key insight: Exit when expensive AND trend weakens")
    elif improvement_50 > 20 or improvement_100 > 20:
        print(f"   ✅ Momentum filters add significant value!")
    elif improvement_50 > 0 or improvement_100 > 0:
        print(f"   ✓ Momentum filters add modest value")
    else:
        print(f"   ❌ Momentum filters don't improve performance")
        print(f"   💡 Even combined with valuation, exits hurt returns")

print(f"\n3. RECOMMENDATION:")

if best['name'] == 'Never Exit':
    print(f"   🎯 Optimal: Use Check's entries, NEVER EXIT")
    print(f"   📝 Even momentum filters don't help")
    print(f"   💡 Bitcoin's long-term trend dominates all exit strategies")
elif 'AND' in best['name']:  # Combined strategy
    print(f"   🎯 Optimal: Use Check's entries + {best['name']}")
    print(f"   📝 Exit only when BOTH overvalued AND momentum weakens")
    print(f"   ✅ This keeps you invested during strong trends!")
else:
    print(f"   🎯 Optimal: {best['name']}")

if best['total_return'] > bh_return:
    print(f"\n   🏆 SUCCESS: Beats buy-and-hold by {best['total_return'] - bh_return:.1f}%!")
else:
    gap = bh_return - best['total_return']
    print(f"\n   ⚠️  Still trails buy-and-hold by {gap:.1f}%")
    if best_sharpe['sharpe'] > never['sharpe']:
        print(f"   But {best_sharpe['name']} has better risk-adjusted returns")

print("\n" + "="*90)

## Conclusion

**Testing momentum-based exit filters to solve the fundamental problem:**

- Pure valuation exits (STH-MVRV > 1.5) fire too early
- Pure momentum exits (Price < 50MA) might be too reactive
- **Combined approach:** Exit when overvalued AND momentum weakens

**Key tests:**
1. Does STH-MVRV > 1.5 + Price < 50MA beat "Never Exit"?
2. Is 50-day MA better than 100-day MA for exits?
3. Does MACD or death cross work better than simple MA?
4. Can we finally beat buy-and-hold?

This approach keeps you invested during strong uptrends (even when expensive) and only exits when the trend actually reverses.